# Geo-FNO Residual + Demean + Random Split Experiment\n\nNotebook version of `geo_FNO_train_test_residual_demean.py`.\nThis keeps existing files untouched and runs as a standalone experiment.\n

In [ ]:
"""
Geo-FNO train+test experiment with least-invasive robustness changes:
1) residual target toggle
2) per-sample input-mean demeaning toggle
3) randomized train/val subset sampling

This file is standalone and does not modify existing training files.
"""

import json
import os
from dataclasses import dataclass, asdict
from typing import Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, Subset

from geo_FNO_def import FNO2d, IPHI, get_global_L_from_h5


def fit_circle_kasa(x: np.ndarray, y: np.ndarray) -> Tuple[float, float, float]:
    A = np.stack([x, y, np.ones_like(x)], axis=1)
    b = -(x ** 2 + y ** 2)
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    a, b_, c = sol
    xc = -a / 2.0
    yc = -b_ / 2.0
    r2 = (a * a + b_ * b_) / 4.0 - c
    r = float(np.sqrt(max(r2, 0.0)))
    return float(xc), float(yc), float(r)


def estimate_cylinder_from_label6(
    pos: np.ndarray,
    node_type: np.ndarray,
    boundary_label: int = 6,
    band_frac: float = 0.06,
) -> Tuple[float, float, float]:
    xy = pos[node_type == boundary_label]
    if xy.shape[0] < 20:
        raise RuntimeError(f"Too few nodes with node_type=={boundary_label}")

    y = xy[:, 1]
    ymin, ymax = float(y.min()), float(y.max())
    band = band_frac * (ymax - ymin)
    cyl_mask = (y > ymin + band) & (y < ymax - band)
    xy_cyl = xy[cyl_mask]
    if xy_cyl.shape[0] < 20:
        raise RuntimeError(
            "After wall-band filtering, too few points to fit circle. "
            f"Try smaller band_frac. points={xy_cyl.shape[0]}"
        )

    return fit_circle_kasa(xy_cyl[:, 0], xy_cyl[:, 1])


class CylinderFlowGeoFNODataset(Dataset):
    def __init__(
        self,
        h5_path: str,
        t_in: int = 0,
        t_out: int = -1,
        boundary_label: int = 6,
        band_frac: float = 0.06,
        cache_geom: bool = True,
    ):
        super().__init__()
        self.f = h5py.File(h5_path, "r")
        self.keys = sorted([k for k in self.f.keys() if k.startswith("sample_")])
        self.t_in = t_in
        self.t_out = t_out
        self.boundary_label = boundary_label
        self.band_frac = band_frac
        self.cache_geom = cache_geom
        self._geom_cache = {}

    def __len__(self):
        return len(self.keys)

    def _get_geom(self, key: str) -> Tuple[float, float, float]:
        if self.cache_geom and key in self._geom_cache:
            return self._geom_cache[key]
        g = self.f[key]
        pos = g["pos"][:]
        node_type = g["node_type"][:]
        xc, yc, r = estimate_cylinder_from_label6(
            pos, node_type, boundary_label=self.boundary_label, band_frac=self.band_frac
        )
        if self.cache_geom:
            self._geom_cache[key] = (xc, yc, r)
        return xc, yc, r

    def __getitem__(self, idx: int):
        key = self.keys[idx]
        g = self.f[key]
        pos = g["pos"][:].astype(np.float32)
        vel = g["vel"][:]
        u_in = vel[self.t_in].astype(np.float32)
        u_out = vel[self.t_out].astype(np.float32)

        xc, yc, r = self._get_geom(key)
        code = np.zeros((42,), dtype=np.float32)
        code[0], code[1], code[2] = xc, yc, r

        return (
            torch.from_numpy(pos),
            torch.from_numpy(u_in),
            torch.from_numpy(u_out),
            torch.from_numpy(code),
        )


def collate_bs1(batch):
    return batch[0]


def random_subset_indices(total: int, n: int, seed: int):
    n = min(n, total)
    rng = np.random.default_rng(seed)
    return rng.choice(total, size=n, replace=False).tolist()


def compute_io_stats_from_indices(ds, indices):
    sum_in = torch.zeros(2, dtype=torch.float64)
    sq_in = torch.zeros(2, dtype=torch.float64)
    n_in = 0

    sum_out = torch.zeros(2, dtype=torch.float64)
    sq_out = torch.zeros(2, dtype=torch.float64)
    n_out = 0

    for i in indices:
        _, u_in, u_out, _ = ds[i]
        u_in64 = u_in.to(torch.float64)
        u_out64 = u_out.to(torch.float64)
        sum_in += u_in64.sum(dim=0)
        sq_in += (u_in64 * u_in64).sum(dim=0)
        n_in += int(u_in64.shape[0])
        sum_out += u_out64.sum(dim=0)
        sq_out += (u_out64 * u_out64).sum(dim=0)
        n_out += int(u_out64.shape[0])

    mean_in = sum_in / max(1, n_in)
    mean_out = sum_out / max(1, n_out)
    var_in = torch.clamp(sq_in / max(1, n_in) - mean_in * mean_in, min=1e-12)
    var_out = torch.clamp(sq_out / max(1, n_out) - mean_out * mean_out, min=1e-12)
    std_in = torch.sqrt(var_in)
    std_out = torch.sqrt(var_out)
    return {
        "u_in_mean": mean_in.to(torch.float32),
        "u_in_std": std_in.to(torch.float32),
        "u_out_mean": mean_out.to(torch.float32),
        "u_out_std": std_out.to(torch.float32),
    }


def preprocess_sample(
    u_in: torch.Tensor,
    u_out: torch.Tensor,
    use_residual_target: bool,
    use_sample_mean_demean: bool,
):
    """
    u_in/u_out: (B,N,2)
    Returns:
      u_in_proc, target_proc, sample_mean_in
    """
    sample_mean_in = u_in.mean(dim=1, keepdim=True)  # (B,1,2)

    if use_sample_mean_demean:
        u_in_proc = u_in - sample_mean_in
    else:
        u_in_proc = u_in

    if use_residual_target:
        target_proc = u_out - u_in
    else:
        if use_sample_mean_demean:
            target_proc = u_out - sample_mean_in
        else:
            target_proc = u_out

    return u_in_proc, target_proc, sample_mean_in


def reconstruct_u_out_pred(
    pred_proc: torch.Tensor,
    u_in_raw: torch.Tensor,
    sample_mean_in: torch.Tensor,
    use_residual_target: bool,
    use_sample_mean_demean: bool,
):
    if use_residual_target:
        return u_in_raw + pred_proc
    if use_sample_mean_demean:
        return pred_proc + sample_mean_in
    return pred_proc


@dataclass
class TrainConfig:
    train_h5: str = "/scratch/mnhagen/datasets/incompressible_euler/train.h5"
    test_h5: str = "/scratch/mnhagen/datasets/incompressible_euler/test.h5"
    ntrain: int = 1000
    ntest: int = 200
    random_train_subset: bool = True
    random_test_subset: bool = True
    split_seed: int = 0

    t_in: int = 0
    t_out: int = -1
    boundary_label: int = 6
    band_frac: float = 0.06
    batch_size: int = 1
    epochs: int = 1000
    learning_rate_fno: float = 1e-3
    learning_rate_iphi: float = 1e-4
    modes: int = 20
    width: int = 32
    s1: int = 80
    s2: int = 40
    device: str = "cuda:0" if torch.cuda.is_available() else "cpu"
    val_patience: int = 25
    improve_thresh: float = 1e-3

    # Least-invasive experiment toggles
    use_residual_target: bool = True
    use_sample_mean_demean: bool = True

    # Keep existing optional model-side normalization path
    use_io_normalization: bool = False

    ckpt_dir: str = "/scratch/mnhagen/models/geofno"
    ckpt_name: str = "cylinder_vel_t0_t-1_residual_demean_rand"


def run_training(cfg: TrainConfig):
    print("Config:")
    print(json.dumps(asdict(cfg), indent=2))

    train_ds = CylinderFlowGeoFNODataset(
        cfg.train_h5,
        t_in=cfg.t_in,
        t_out=cfg.t_out,
        boundary_label=cfg.boundary_label,
        band_frac=cfg.band_frac,
        cache_geom=True,
    )
    test_ds = CylinderFlowGeoFNODataset(
        cfg.test_h5,
        t_in=cfg.t_in,
        t_out=cfg.t_out,
        boundary_label=cfg.boundary_label,
        band_frac=cfg.band_frac,
        cache_geom=True,
    )

    if cfg.random_train_subset:
        train_idx = random_subset_indices(len(train_ds), cfg.ntrain, cfg.split_seed)
    else:
        train_idx = list(range(min(cfg.ntrain, len(train_ds))))

    if cfg.random_test_subset:
        test_idx = random_subset_indices(len(test_ds), cfg.ntest, cfg.split_seed + 1)
    else:
        test_idx = list(range(min(cfg.ntest, len(test_ds))))

    train_loader = DataLoader(
        Subset(train_ds, train_idx),
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_bs1,
    )
    test_loader = DataLoader(
        Subset(test_ds, test_idx),
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_bs1,
    )

    print("Train subset size:", len(train_idx))
    print("Test subset size:", len(test_idx))

    L_global, key_used = get_global_L_from_h5(cfg.train_h5)
    print("Using L_global from", key_used, ":", L_global)

    model = FNO2d(
        cfg.modes,
        cfg.modes,
        cfg.width,
        in_channels=2,
        out_channels=2,
        is_mesh=False,
        s1=cfg.s1,
        s2=cfg.s2,
        L=L_global,
    ).to(cfg.device)
    model_iphi = IPHI(width=32, device=cfg.device).to(cfg.device)

    if cfg.use_io_normalization:
        stats = compute_io_stats_from_indices(train_ds, train_idx)
        model.set_io_normalization(
            stats["u_in_mean"],
            stats["u_in_std"],
            stats["u_out_mean"],
            stats["u_out_std"],
            enabled=True,
        )
        print("I/O normalization enabled.")
    else:
        model.set_io_normalization_enabled(False)
        print("I/O normalization disabled.")

    opt_fno = Adam(model.parameters(), lr=cfg.learning_rate_fno, weight_decay=1e-4)
    opt_iphi = Adam(model_iphi.parameters(), lr=cfg.learning_rate_iphi, weight_decay=1e-4)
    sched_fno = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fno, T_max=cfg.epochs)
    sched_iphi = torch.optim.lr_scheduler.CosineAnnealingLR(opt_iphi, T_max=cfg.epochs)
    loss_fn = nn.MSELoss()

    best_val_loss = 1e12
    epochs_no_improve = 0
    best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}

    for ep in range(cfg.epochs):
        model.train()
        model_iphi.train()
        train_loss_proc = 0.0
        train_loss_phys = 0.0

        for pos, u_in, u_out, code42 in train_loader:
            pos = pos.unsqueeze(0).to(cfg.device)
            u_in = u_in.unsqueeze(0).to(cfg.device)
            u_out = u_out.unsqueeze(0).to(cfg.device)
            code42 = code42.unsqueeze(0).to(cfg.device)

            u_in_proc, target_proc, mean_in = preprocess_sample(
                u_in=u_in,
                u_out=u_out,
                use_residual_target=cfg.use_residual_target,
                use_sample_mean_demean=cfg.use_sample_mean_demean,
            )

            opt_fno.zero_grad()
            opt_iphi.zero_grad()

            pred_proc = model(u_in_proc, code=code42, x_in=pos, x_out=pos, iphi=model_iphi)
            loss_proc = loss_fn(pred_proc, target_proc)
            loss_proc.backward()

            opt_fno.step()
            opt_iphi.step()

            with torch.no_grad():
                u_pred = reconstruct_u_out_pred(
                    pred_proc=pred_proc,
                    u_in_raw=u_in,
                    sample_mean_in=mean_in,
                    use_residual_target=cfg.use_residual_target,
                    use_sample_mean_demean=cfg.use_sample_mean_demean,
                )
                loss_phys = loss_fn(u_pred, u_out)

            train_loss_proc += float(loss_proc.item())
            train_loss_phys += float(loss_phys.item())

        train_loss_proc /= max(1, len(train_loader))
        train_loss_phys /= max(1, len(train_loader))

        model.eval()
        model_iphi.eval()
        val_loss_proc = 0.0
        val_loss_phys = 0.0
        with torch.no_grad():
            for pos, u_in, u_out, code42 in test_loader:
                pos = pos.unsqueeze(0).to(cfg.device)
                u_in = u_in.unsqueeze(0).to(cfg.device)
                u_out = u_out.unsqueeze(0).to(cfg.device)
                code42 = code42.unsqueeze(0).to(cfg.device)

                u_in_proc, target_proc, mean_in = preprocess_sample(
                    u_in=u_in,
                    u_out=u_out,
                    use_residual_target=cfg.use_residual_target,
                    use_sample_mean_demean=cfg.use_sample_mean_demean,
                )

                pred_proc = model(u_in_proc, code=code42, x_in=pos, x_out=pos, iphi=model_iphi)
                loss_proc = loss_fn(pred_proc, target_proc)
                u_pred = reconstruct_u_out_pred(
                    pred_proc=pred_proc,
                    u_in_raw=u_in,
                    sample_mean_in=mean_in,
                    use_residual_target=cfg.use_residual_target,
                    use_sample_mean_demean=cfg.use_sample_mean_demean,
                )
                loss_phys = loss_fn(u_pred, u_out)

                val_loss_proc += float(loss_proc.item())
                val_loss_phys += float(loss_phys.item())

        val_loss_proc /= max(1, len(test_loader))
        val_loss_phys /= max(1, len(test_loader))

        sched_fno.step()
        sched_iphi.step()

        print(
            f"ep={ep:04d} "
            f"train_proc={train_loss_proc:.6e} train_phys={train_loss_phys:.6e} "
            f"val_proc={val_loss_proc:.6e} val_phys={val_loss_phys:.6e}"
        )

        if val_loss_phys < best_val_loss * (1 - cfg.improve_thresh):
            best_val_loss = val_loss_phys
            best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve = 0
        else:
            if best_val_loss * (1 - cfg.improve_thresh) < val_loss_phys < best_val_loss:
                best_val_loss = val_loss_phys
                best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve += 1
            if epochs_no_improve >= cfg.val_patience:
                print("Early stop triggered.")
                break

    model.load_state_dict(best_state_model)
    model_iphi.load_state_dict(best_state_iphi)
    print("Best val physical loss:", best_val_loss)

    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    model_path = os.path.join(cfg.ckpt_dir, f"{cfg.ckpt_name}_fno.pt")
    iphi_path = os.path.join(cfg.ckpt_dir, f"{cfg.ckpt_name}_iphi.pt")
    meta_path = os.path.join(cfg.ckpt_dir, f"{cfg.ckpt_name}_meta.json")
    torch.save(model.state_dict(), model_path)
    torch.save(model_iphi.state_dict(), iphi_path)

    meta = asdict(cfg)
    meta["best_val_physical_loss"] = best_val_loss
    meta["model_path"] = model_path
    meta["iphi_path"] = iphi_path
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved:", model_path)
    print("Saved:", iphi_path)
    print("Saved:", meta_path)

    return model, model_iphi, meta




In [ ]:
# Run training with defaults in TrainConfig; edit fields before running if needed.
cfg = TrainConfig()
model, model_iphi, meta = run_training(cfg)
